# L7.3 · SmolVLA: la fila del VLM

| Notebook | Qué abre |
|---|---|
| L7.1 | El mapa: qué entra, qué sale, dónde están los 450 M de parámetros |
| L7.2 | La imagen: de la foto a 64 tokens |
| **L7.3** | La fila del VLM: palabras, estado y quién puede mirar a quién |
| L7.4 | Adentro del VLM: qué miran las palabras, por qué 16 de 32 capas, qué queda guardado |
| `06` (flujo) | Entrenarlo con nuestros datos |
| L7.5 | El experto de acciones: la brújula de L6 con ojos |
| L7.6 | Las 10 pasadas: un bloque de verdad pasando de ruido a movimiento |
| L7.7 | El entrenamiento por dentro |
| `07` (flujo) | Evaluación contra ACT, en los mismos episodios |

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import DATASETS_DIR
from toolkit.policy import smolvla_internals as vla
from toolkit.policy.smolvla_g1_base import G1_BASE_DIR, load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"

# Las estadísticas de normalización salen sólo de los episodios de entrenamiento, como al entrenar.
TRAIN = DATASETS_DIR / "local" / "pick_mameluco_act_v1_train"
train_meta = LeRobotDataset("local/pick_mameluco_act_v1_train", root=TRAIN).meta
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1")
names = dataset.meta.features["observation.state"]["names"]
joint_names = names[0] if isinstance(names[0], list) else names
policy, preprocess, _ = load_g1_policy(G1_BASE_DIR, train_meta.stats, device)
model = policy.model
tokenizer = model.vlm_with_expert.processor.tokenizer

frame = dataset[int(dataset.meta.episodes["dataset_from_index"][10]) + 120]  # el instante de L7.1
batch = preprocess(dict(frame))
obs = vla.prepare_observation(policy, batch)

## 1. El problema

El VLM recibe **una sola fila de vectores de 960 números**. Los 64 tokens de la
imagen ya los tenemos (L7.2). Faltan dos cosas que no son imágenes: la
instrucción, que es texto, y el estado del robot, que son 26 ángulos. Cada una
tiene que volverse vectores de 960 para poder sumarse a la fila. Y después hay
que decidir quién puede mirar a quién adentro de esa fila.

## 2. La instrucción: de texto a tokens

Primero, el preprocesador le agrega un salto de línea al final (una costumbre
que SmolVLA heredó del tokenizador de π0). Después, el **tokenizador** corta el
texto en pedazos que están en su vocabulario de 49.280: palabras comunes
enteras, palabras raras en partes. Cada pedazo tiene un número, y la fila se
rellena con un token de relleno hasta 48.

In [ ]:
text = frame["task"] + "\n"
ids = obs.lang_tokens[0].cpu()
real = obs.lang_masks[0].cpu()
pieces = tokenizer.convert_ids_to_tokens(ids.tolist())
print(f"texto: {text!r}")
print(f"{int(real.sum())} tokens de verdad + {int((~real).sum())} de relleno = {len(ids)}\n")
for i, (token_id, piece, is_real) in enumerate(zip(ids.tolist(), pieces, real.tolist())):
    if is_real or i == int(real.sum()):
        shown = piece.replace("Ġ", "·").replace("Ċ", "⏎")
        print(f"  {i:>2}  id {token_id:>6}  {shown!r}{'' if is_real else '   ← y así hasta 48: relleno'}")

(El `·` marca un espacio antes de la palabra, y `⏎` el salto de línea.)

Cada número elige una fila de la **tabla de palabras** (49.280 × 960, la de
L7.1), y esa fila es el vector de la palabra. Se multiplica por √960, igual
que la imagen.

¿Esa tabla sabe algo? Buscamos, para algunas palabras de la instrucción, qué
otras filas de la tabla apuntan en la dirección más parecida:

In [ ]:
table = model.vlm_with_expert.get_vlm_model().text_model.get_input_embeddings().weight.float()
table_unit = torch.nn.functional.normalize(table, dim=1)


def neighbors(word, k=6):
    token_id = tokenizer.convert_tokens_to_ids(word)
    similarity = table_unit @ table_unit[token_id]
    best = similarity.topk(k + 1).indices[1:]
    return [tokenizer.convert_ids_to_tokens(i.item()).replace("Ġ", "") for i in best]


for word in ["Ġshelf", "Ġpick", "Ġfolded"]:
    print(f"{word.replace('Ġ', ''):>8} → {neighbors(word)}")

Los vecinos son casi todos **variantes de la misma palabra**: plurales,
mayúsculas, pedazos. La tabla agrupa por forma más que por significado; el
significado se arma recién en las capas, cuando cada token mira a los demás.

Fijate también que "coverall" no está en el vocabulario: el modelo lo ve como
"cover" + "all".

## 3. El estado: 26 ángulos en un solo token

Los 26 ángulos actuales pasan por tres pasos antes de entrar a la fila:

1. **Normalizar**: a cada articulación se le resta su promedio y se divide por
   su desvío, con las estadísticas de los episodios de entrenamiento. Así todas
   quedan en la misma escala (L4).
2. **Rellenar con ceros hasta 32**: SmolVLA admite robots de hasta 32
   articulaciones, y el resto se completa con ceros.
3. **Una matriz de 32 × 960** los convierte en **un solo vector**: el token del
   estado. Esta matriz sí se entrena.

In [ ]:
raw = frame["observation.state"].numpy()
normalized = batch["observation.state"][0].float().cpu().numpy()
padded = obs.state[0].float().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)
axes[0].barh(range(26), raw, color="tab:orange")
axes[0].set_yticks(range(26), joint_names, fontsize=6)
axes[0].set_title("1 · ángulos (radianes)")
axes[1].barh(range(26), normalized, color="tab:orange")
axes[1].set_yticks(range(26), [""] * 26)
axes[1].set_title("normalizados (desvíos del promedio)")
axes[2].barh(range(32), padded, color=["tab:orange"] * 26 + ["lightgray"] * 6)
axes[2].set_yticks(range(32), [""] * 32)
axes[2].set_title("2 · rellenados con ceros hasta 32")
axes[2].text(0.05, 29, "+ 6 ceros", color="gray")
for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
with torch.no_grad():
    state_token = model.state_proj(obs.state)
print("3 · matriz:", tuple(model.state_proj.weight.T.shape), "→ token del estado:", tuple(state_token.shape))

Normalizar cambia mucho el cuadro: los dedos, que en radianes casi no se ven,
pasan a pesar tanto como el codo (L4 explica por qué importa). El relleno son 6
ceros al final. Y todo el estado del cuerpo termina en un solo token, igual que
en ACT.

La matriz de 32 × 960 de `smolvla_base` se aprendió para brazos SO-100; al
entrenar con nuestros datos se reaprende.

## 4. La fila completa

Se pegan las tres partes: imagen, texto y estado. Cada token recibe además una
**posición** (en qué lugar de la fila está), que el VLM usa para saber el
orden. Los tokens de relleno no avanzan el contador: el estado queda en la
posición siguiente a la última palabra de verdad, no en la 112.

In [ ]:
with torch.no_grad():
    prefix = vla.embed_prefix(model, obs)
positions = prefix.position_ids[0].cpu()
segment_of = np.empty(prefix.embs.shape[1], dtype=object)
for name, s in prefix.segments.items():
    segment_of[s] = name
print("la fila:", tuple(prefix.embs.shape), "→", {n: s.stop - s.start for n, s in prefix.segments.items()})
print("posición del token de estado:", int(positions[prefix.segments["estado"]][0]))

fig, ax = plt.subplots(figsize=(13, 2.8))
colors = {"imagen": "tab:blue", "texto": "tab:green", "estado": "tab:orange"}
pad = ~prefix.pad_masks[0].cpu()
ax.scatter(range(len(positions)), positions, c=[("lightgray" if p else colors[s]) for s, p in zip(segment_of, pad)], s=12)
ax.set_xlabel("lugar en la fila")
ax.set_ylabel("posición")
ax.set_title("posiciones: el relleno (gris) no avanza el contador")
plt.tight_layout()

## 5. Quién puede mirar a quién

Adentro de cada capa, cada token arma su nueva versión mirando a otros tokens
(atención). Una **máscara** decide a quiénes puede mirar. En un modelo de chat,
como el SmolLM2 que vive adentro del VLM, cada palabra mira sólo hacia atrás:
así se entrena para escribir de a una.

SmolVLA usa otra regla, que viene de π0. Cada token tiene un número, 0 o 1, y
se suman de izquierda a derecha; un token puede mirar a otro si la suma del
otro no es mayor que la suya, y ninguno de los dos es relleno. Imagen y texto
llevan 0, el estado lleva 1:

In [ ]:
print("números de la regla:", {n: sorted(set(prefix.att_masks[0, s].int().tolist())) for n, s in prefix.segments.items()})
mask = prefix.att_2d[0].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
causal = np.tril(np.ones_like(mask))
for ax, m, title in [(axes[0], causal, "un modelo de chat: sólo hacia atrás"),
                     (axes[1], mask, "la fila de SmolVLA")]:
    ax.imshow(m, cmap="Greys", interpolation="nearest")
    for name, s in prefix.segments.items():
        ax.axhline(s.start - 0.5, color=colors[name], lw=1)
        ax.axvline(s.start - 0.5, color=colors[name], lw=1)
    ax.set_xlabel("a quién mira")
    ax.set_ylabel("quién mira")
    ax.set_title(title)
for name, s in prefix.segments.items():
    axes[1].text((s.start + s.stop) / 2, -3, name, ha="center", color=colors[name], fontsize=9)
state_row = prefix.segments["estado"].start
axes[1].annotate("el estado mira todo (última fila)", (40, state_row), (30, 95), color="tab:orange",
                 arrowprops=dict(arrowstyle="->", color="tab:orange"))
axes[1].annotate("nadie mira al estado\n(última columna, vacía)", (state_row, 40), (78, 20), color="tab:orange",
                 arrowprops=dict(arrowstyle="->", color="tab:orange"))
axes[1].text(80, 60, "relleno: ni mira\nni es mirado", color="gray")
fig.suptitle("negro = puede mirar")
plt.tight_layout()

Tres cosas:

- **Imagen y texto se miran en las dos direcciones** (el bloque negro
  grande): una palabra puede mirar a la imagen, y la imagen a las palabras,
  sin importar el orden. El SmolLM2 de adentro se entrenó mirando sólo hacia
  atrás; acá se lo usa distinto, como hace π0.
- **El estado mira todo, y nadie mira al estado.** El código lo dice con un
  comentario: que la imagen y el lenguaje no miren al estado ni a las acciones.
- **El relleno ni mira ni es mirado.**

El paper probó poner el estado acá, del lado del VLM, contra ponerlo del lado
del experto, como hace π0. Del lado del VLM le fue mejor: 80,3 % contra 73,3 %
de éxito en LIBERO, con la variante de atención cruzada (tabla 11).

## Lo que nos llevamos

- La instrucción se corta en tokens del vocabulario del VLM, cada uno toma su
  vector de la tabla de palabras, y la fila se rellena hasta 48.
- El estado se normaliza, se rellena con ceros hasta 32 y una matriz lo vuelve
  **un solo token**. Esa matriz se entrena; la tabla de palabras no.
- La fila del VLM es [64 imagen | 48 texto | 1 estado], con posiciones que
  saltean el relleno.
- La máscara: imagen y texto se miran entre sí en las dos direcciones, el
  estado mira todo, y nadie mira al estado.

En L7.4 metemos esta fila en las 16 capas del VLM y miramos qué mira cada
palabra en la imagen.

## Referencias

- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025), sección
  3.1 y tabla 11 (estado del lado del VLM contra del lado del experto).
- Black et al., [π0](https://arxiv.org/abs/2410.24164) (2024): la regla de la
  máscara por bloques.
- Código: `embed_prefix` y `make_att_2d_masks` en
  `lerobot/policies/smolvla/modeling_smolvla.py` (LeRobot 0.4.1).